# Object Reconstruction Quality Ranking with LineUp

This notebook ranks objects in a 3DGS scene by reconstruction quality (PSNR, SSIM),
comparing rendered views against ground truth on segmented object regions.

**Two modes:**
- **Mode A (fast):** Load pre-computed metrics from `.pkl` files (run Section 2A, skip 2B, 3)
- **Mode B (full):** Compute metrics from scratch using LangSAM segmentation (skip 2A, run 2B)

In [1]:
import sys
import argparse
from pathlib import Path

import torch
import numpy as np
import cv2
import pandas as pd

gs_root = Path("gaussian-splatting").resolve()
sys.path.insert(0, str(gs_root))

from arguments import ModelParams, PipelineParams
from scene import Scene, GaussianModel
from gaussian_renderer import render
from utils.loss_utils import l1_loss, ssim
from utils.image_utils import psnr

try:
    from fused_ssim import fused_ssim
    FUSED_SSIM_AVAILABLE = True
except ImportError:
    FUSED_SSIM_AVAILABLE = False

device = "cuda:0"

## 1. Load Scene

In [2]:
source_path = Path("gaussian-splatting/data/images1/kitchen").resolve()
model_path  = Path("gaussian-splatting/models/kitchen").resolve()
ITERATION   = 30000

parser = argparse.ArgumentParser()
lp = ModelParams(parser)
pp = PipelineParams(parser)

args = parser.parse_args([
    "--source_path", str(source_path),
    "--model_path",  str(model_path),
    "--images",      "images",
])

dataset = lp.extract(args)
pipe    = pp.extract(args)

gaussians = GaussianModel(dataset.sh_degree, optimizer_type="adam")
scene = Scene(dataset, gaussians, load_iteration=ITERATION, shuffle=False, resolution_scales=[1.0])

train_cams = scene.getTrainCameras(scale=1.0)
print(f"Loaded {len(train_cams)} cameras")

bg_color = [1, 1, 1] if dataset.white_background else [0, 0, 0]
background = torch.tensor(bg_color, dtype=torch.float32, device=device)

Loading trained model at iteration 30000
Reading camera 279/279
Loading Training Cameras
[ INFO ] Encountered quite large input images (>1.6K pixels width), rescaling to 1.6K.
 If this is not desired, please explicitly specify '--resolution/-r' as 1
Loading Test Cameras
Loaded 279 cameras


## 2A. Load Pre-computed Metrics from PKL Files (fast path)

Load `.pkl` files produced by your teammates. Each pkl is a DataFrame with per-view
metrics for one object. Map object names to file paths below.

**Run this section OR Section 2B, not both.**

In [3]:
import pandas as pd

SCENE_NAME = "kitchen"
PKL_PATH = f"metrics/{SCENE_NAME}.pkl"

df_raw = pd.read_pickle(PKL_PATH)

df_per_view = df_raw.rename(columns={
    "image_name": "view",
    "PSNR_dB": "psnr",
    "SSIM": "ssim",
    "total_pixels": "pixel_count",
})[["object", "view", "psnr", "ssim", "pixel_count"]]

print(f"Loaded {len(df_per_view)} measurements across {df_per_view['object'].nunique()} objects")
print(f"Objects: {df_per_view['object'].unique().tolist()}")
df_per_view.head(10)

Loaded 837 measurements across 3 objects
Objects: ['tractor', 'red mat', 'blue mat']


,object,view,psnr,ssim,pixel_count
0,tractor,DSCF0656.JPG,33.861767,0.975383,264391
1,tractor,DSCF0657.JPG,34.514725,0.979243,260122
2,tractor,DSCF0658.JPG,35.477566,0.981197,265526
3,tractor,DSCF0659.JPG,36.015808,0.980923,269035
4,tractor,DSCF0660.JPG,37.075344,0.983315,273033
5,tractor,DSCF0661.JPG,36.726814,0.982546,279480
6,tractor,DSCF0662.JPG,35.477055,0.981641,274322
7,tractor,DSCF0663.JPG,35.053719,0.981478,280633
8,tractor,DSCF0664.JPG,28.523878,0.964536,283641
9,tractor,DSCF0665.JPG,35.155769,0.981011,280197


## 2B. Compute from Scratch with LangSAM (full pipeline)

**Skip this section if you loaded from PKL files above.**

Runs LangSAM segmentation + 3DGS rendering for each prompt.

In [ ]:
import contextlib, io
from PIL import Image
from lang_sam import LangSAM

OBJECT_PROMPTS = ["tractor", "potted plant", "clay pot", "wooden table", "wall"]


def quiet_predict(model, images_pil, texts_prompt):
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
        return model.predict(images_pil, texts_prompt)


def predict_masks_safe_batch(langsam_model, images_pil, texts_prompt):
    assert len(images_pil) == len(texts_prompt)
    try:
        return list(quiet_predict(langsam_model, images_pil, texts_prompt))
    except (AssertionError, Exception):
        results = []
        for img_pil, txt in zip(images_pil, texts_prompt):
            try:
                single_res = quiet_predict(langsam_model, [img_pil], [txt])
                results.append(single_res[0])
            except Exception:
                results.append(None)
                torch.cuda.empty_cache()
        return results


langsam_model = LangSAM(device=device)

gt_pils = []
gt_names = []
for cam in train_cams:
    gt = torch.clamp(cam.original_image.cpu(), 0.0, 1.0)
    gt_np = (gt.permute(1, 2, 0).numpy() * 255).clip(0, 255).astype(np.uint8)
    gt_pils.append(Image.fromarray(gt_np))
    gt_names.append(cam.image_name)

print(f"Prepared {len(gt_pils)} images for segmentation")

In [ ]:
from tqdm.notebook import tqdm

BATCH_SIZE = 2
# all_masks[prompt][image_name] = (H, W) uint8 mask, or None
all_masks = {}

for prompt in OBJECT_PROMPTS:
    print(f"\n── Segmenting: \"{prompt}\" ──")
    masks_by_name = {}

    for i in tqdm(range(0, len(gt_pils), BATCH_SIZE), desc=prompt):
        batch_imgs  = gt_pils[i:i+BATCH_SIZE]
        batch_names = gt_names[i:i+BATCH_SIZE]
        batch_texts = [prompt] * len(batch_imgs)

        batch_results = predict_masks_safe_batch(langsam_model, batch_imgs, batch_texts)

        for name, res in zip(batch_names, batch_results):
            if res is None:
                masks_by_name[name] = None
                continue

            masks = res["masks"]
            if isinstance(masks, torch.Tensor):
                masks_np = masks.cpu().numpy()
            else:
                masks_np = np.asarray(masks)

            if masks_np.shape[0] == 0:
                masks_by_name[name] = None
                continue

            masks_by_name[name] = (masks_np[0] > 0.5).astype(np.uint8)

        torch.cuda.empty_cache()

    n_found = sum(1 for v in masks_by_name.values() if v is not None)
    print(f"  Found \"{prompt}\" in {n_found}/{len(masks_by_name)} views")
    all_masks[prompt] = masks_by_name

del langsam_model
torch.cuda.empty_cache()
print("\nLangSAM unloaded, VRAM freed for rendering.")

## 3. Compute Per-Object Per-View Metrics (2B only)

**Skip if you used Section 2A.** `df_per_view` is already populated from the pkl files.

In [ ]:
def compute_masked_metrics(rendered, gt, mask_bool):
    """
    Compute PSNR and SSIM on the masked region only.

    Args:
        rendered: (3, H, W) tensor, values in [0, 1]
        gt:       (3, H, W) tensor, values in [0, 1]
        mask_bool: (H, W) numpy bool array
    Returns:
        (psnr_val, ssim_val, pixel_count)
    """
    mask_t = torch.from_numpy(mask_bool).to(rendered.device).unsqueeze(0)  # (1, H, W)

    rendered_masked = rendered * mask_t
    gt_masked = gt * mask_t

    pixel_count = mask_bool.sum()
    if pixel_count < 10:
        return None, None, 0

    mse = ((rendered_masked - gt_masked) ** 2).sum() / (pixel_count * 3)
    psnr_val = 10 * torch.log10(1.0 / mse).item() if mse > 0 else 100.0

    if FUSED_SSIM_AVAILABLE:
        ssim_val = fused_ssim(
            rendered_masked.unsqueeze(0),
            gt_masked.unsqueeze(0)
        ).item()
    else:
        ssim_val = ssim(rendered_masked, gt_masked).item()

    return psnr_val, ssim_val, int(pixel_count)

In [ ]:
records = []

with torch.no_grad():
    for cam_idx, cam in enumerate(tqdm(train_cams, desc="Rendering views")):
        pkg = render(cam, scene.gaussians, pipe, background)
        rendered = torch.clamp(pkg["render"], 0.0, 1.0)
        gt = torch.clamp(cam.original_image.to(device), 0.0, 1.0)

        for prompt in OBJECT_PROMPTS:
            mask = all_masks[prompt].get(cam.image_name)
            if mask is None:
                continue
            mask_bool = mask.astype(bool)
            p, s, px = compute_masked_metrics(rendered, gt, mask_bool)
            if p is not None:
                records.append({
                    "object": prompt,
                    "view": cam.image_name,
                    "view_idx": cam_idx,
                    "psnr": p,
                    "ssim": s,
                    "pixel_count": px,
                })

df_per_view = pd.DataFrame(records)
print(f"Collected {len(df_per_view)} measurements across {df_per_view['object'].nunique()} objects")
df_per_view.head(10)

## 4. Aggregate: Weighted Average per Object

Weight each view's metric by `pixel_count` — views where the object is larger
contribute more to the final score.

In [4]:
ALPHA = 0.5  # weight between PSNR and SSIM for combined score


def weighted_mean(group, value_col, weight_col="pixel_count"):
    w = group[weight_col].values.astype(float)
    v = group[value_col].values
    return np.average(v, weights=w)


agg_rows = []
for obj_name, grp in df_per_view.groupby("object"):
    w_psnr = weighted_mean(grp, "psnr")
    w_ssim = weighted_mean(grp, "ssim")

    psnr_min, psnr_max = df_per_view["psnr"].min(), df_per_view["psnr"].max()
    ssim_min, ssim_max = df_per_view["ssim"].min(), df_per_view["ssim"].max()

    psnr_norm = (w_psnr - psnr_min) / (psnr_max - psnr_min + 1e-8)
    ssim_norm = (w_ssim - ssim_min) / (ssim_max - ssim_min + 1e-8)
    combined  = ALPHA * psnr_norm + (1 - ALPHA) * ssim_norm

    agg_rows.append({
        "object": obj_name,
        "psnr": round(w_psnr, 2),
        "ssim": round(w_ssim, 4),
        "combined_score": round(combined, 4),
        "num_views": len(grp),
        "avg_pixel_count": int(grp["pixel_count"].mean()),
    })

df_objects = pd.DataFrame(agg_rows).sort_values("combined_score", ascending=False)
df_objects

,object,psnr,ssim,combined_score,num_views,avg_pixel_count
1,red mat,39.81,0.9781,0.6804,279,390042
2,tractor,38.06,0.9863,0.6730,279,209620
0,blue mat,35.58,0.9739,0.5634,279,413891


## 5. LineUp Ranking Visualization

In [6]:
import lineup_widget

w = lineup_widget.LineUpWidget(
    df_objects,
    options=dict(rowHeight=28),
)
w

LineUpWidget(value=[], layout=Layout(align_self='stretch', height='600px'), options={'rowHeight': 28})

### Analysis: Object-Level Ranking

The LineUp ranking reveals a clear quality hierarchy among the three segmented objects in the kitchen scene:

| Rank | Object   | PSNR (dB) | SSIM   | Combined Score |
|------|----------|-----------|--------|----------------|
| 1    | Red mat  | 39.81     | 0.9781 | 0.6804         |
| 2    | Tractor  | 38.06     | 0.9863 | 0.6730         |
| 3    | Blue mat | 35.58     | 0.9739 | 0.5634         |


- **Red mat** ranks highest overall. Its large, flat, uniformly-colored surface is geometrically simple for the Gaussian representation, yielding the best PSNR (39.81 dB).
- **Tractor** ranks second in combined score having a slightly lower PSNR than the red mat. It achieves the highest SSIM (0.9863) of all objects, meaning that its fine structural details are well-preserved even if pixel-level error is slightly higher.
- **Blue mat** ranks last with a notably lower combined score (0.5634 vs. ~0.68 for the other two). Its PSNR is roughly 4 dB below the red mat, and visual inspection confirms noticeable color inaccuracies in the reconstruction.

The gap between the top two objects and the blue mat is substantial, making it a clear candidate for targeted improvement (e.g., additional training views or object level fine-tuning).

## 6. Per-View Breakdown (optional)

Drill down into per-view metrics for a specific object.

In [7]:
OBJECT_TO_INSPECT = df_per_view["object"].unique()[0]

df_drill = df_per_view[df_per_view["object"] == OBJECT_TO_INSPECT].copy()
df_drill = df_drill.sort_values("psnr", ascending=False)

drill_widget = lineup_widget.LineUpWidget(
    df_drill[["view", "psnr", "ssim", "pixel_count"]],
    options=dict(rowHeight=24),
)
drill_widget

LineUpWidget(value=[], layout=Layout(align_self='stretch', height='600px'), options={'rowHeight': 24})

### Analysis: Per-View Breakdown

The per-view drill-down exposes how reconstruction quality varies across the 279 camera viewpoints for the selected object. Follwing are noticable findings:

- **High variance across views:** PSNR ranges from the 20s on the worst views up to the 40s on the best, indicating that the reconstruction is strongly view-dependent rather than uniformly good or bad.
- **Outlier views:** A small number of viewpoints exhibit sharply lower PSNR (e.g., DSCF0664 at ~28.5 dB for the tractor), suggesting specific camera angles where the Gaussian coverage is insufficient or lighting changes cause degradation. These are prime candidates for additional image capture.
- **SSIM is more stable than PSNR:** SSIM remains above 0.96 for most views, indicating that structural similarity is maintained even when pixel-level error varies. This suggests the failures are primarily color/intensity mismatches rather than geometric distortions.

This view-level analysis supports GUIDE-3D's goal of identifying the specific viewpoints responsible for quality degradation, enabling users to identify viewpoints to re-capture.